In [1]:
# ==============================================================================
# 1. INSTALLATIONS & IMPORTS
# ==============================================================================
!pip install -q transformers datasets evaluate accelerate scikit-learn gensim requests

import os
import requests
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from gensim.models import KeyedVectors

# ==============================================================================
# 2. LABEL MAPPING (GoEmotions & BRIGHTER -> 6 Ekman Classes)
# ==============================================================================
GOEMOTIONS_TO_EKMAN = {
    0: 0, 1: 0, 4: 0, 5: 0, 8: 0, 13: 0, 15: 0, 17: 0, 18: 0, 20: 0, 21: 0, 23: 0, # Joy (ID: 0)
    2: 1, 3: 1, 10: 1,                                                             # Anger (ID: 1)
    9: 2, 12: 2, 16: 2, 24: 2, 25: 2,                                              # Sadness (ID: 2)
    14: 3, 19: 3,                                                                  # Fear (ID: 3)
    11: 4,                                                                         # Disgust (ID: 4)
    6: 5, 7: 5, 22: 5, 26: 5                                                       # Surprise (ID: 5)
}

EKMAN_STRING_TO_ID = {
    "joy": 0, "anger": 1, "sadness": 2, "fear": 3, "disgust": 4, "surprise": 5
}

ekman_id_to_string = {0: "Joy", 1: "Anger", 2: "Sadness", 3: "Fear", 4: "Disgust", 5: "Surprise"}

# ==============================================================================
# 3. LOAD & PREPROCESS TRAINING DATA (GoEmotions - English)
# ==============================================================================
print("Downloading full GoEmotions dataset (English)...")
goemotions = load_dataset("go_emotions", "simplified")

def preprocess_goemotions(example):
    raw_label = example['labels'][0]
    if raw_label in GOEMOTIONS_TO_EKMAN:
        return {"text": example['text'], "label": GOEMOTIONS_TO_EKMAN[raw_label]}
    return {"text": example['text'], "label": -1}

full_train_data = goemotions["train"].map(preprocess_goemotions)
train_data = full_train_data.filter(lambda x: x["label"] != -1).remove_columns(["labels", "id"])
print(f"GoEmotions Training Data Ready: {len(train_data)} sentences.")

# ==============================================================================
# 4. LOAD & PREPROCESS EVALUATION DATA (BRIGHTER - Indonesian)
# ==============================================================================
print("\nFetching official BRIGHTER Indonesian dataset from Hugging Face...")
brighter_ind = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", "ind")
test_data = brighter_ind["test"]

def preprocess_brighter(example):
    if len(example['emotions']) > 0:
        primary_emotion = example['emotions'][0].lower()
        if primary_emotion in EKMAN_STRING_TO_ID:
            return {"text": example['text'], "label": EKMAN_STRING_TO_ID[primary_emotion]}
    return {"text": example['text'], "label": -1}

full_test_data = test_data.map(preprocess_brighter)
cols_to_remove = [c for c in test_data.column_names if c not in ["text", "label"]]
test_dataset = full_test_data.filter(lambda x: x["label"] != -1).remove_columns(cols_to_remove)
print(f"Evaluation Data Ready: {len(test_dataset)} Indonesian sentences.")

# ==============================================================================
# 5. BASELINE MODEL: FASTTEXT (STATIC EMBEDDINGS)
# ==============================================================================
print("\n--- Starting Comparative Analysis: FastText Baseline ---")

def download_muse(lang):
    filename = f"wiki.{lang}.align.vec"
    url = f"https://dl.fbaipublicfiles.com/fasttext/vectors-aligned/{filename}"

    if not os.path.exists(filename):
        print(f"Downloading FastText vectors for '{lang}'... (This takes a moment)")
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()

        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    return filename

en_vec_file = download_muse("en")
ind_vec_file = download_muse("id")

print("Loading FastText vectors into memory...")
en_model = KeyedVectors.load_word2vec_format(en_vec_file, limit=100000)
ind_model = KeyedVectors.load_word2vec_format(ind_vec_file, limit=100000)

def get_sentence_vector(text, w2v_model):
    words = text.lower().split()
    vectors = [w2v_model[w] for w in words if w in w2v_model]
    if len(vectors) == 0:
        return np.zeros(300)
    return np.mean(vectors, axis=0)

print("Training FastText Logistic Regression Head on English...")
X_train_ft = np.array([get_sentence_vector(text, en_model) for text in train_data['text']])
y_train_ft = np.array(train_data['label'])

fasttext_classifier = LogisticRegression(max_iter=1000)
fasttext_classifier.fit(X_train_ft, y_train_ft)

print("Testing FastText directly on Indonesian (Zero-Shot)...")
X_test_ft = np.array([get_sentence_vector(text, ind_model) for text in test_dataset['text']])
y_test_ft = np.array(test_dataset['label'])

ft_predictions = fasttext_classifier.predict(X_test_ft)

print("\n[FastText Performance Metrics - Indonesian]")
print(classification_report(y_test_ft, ft_predictions, zero_division=0))

# ==============================================================================
# 6. ADVANCED MODEL: mBERT (CONTEXTUAL EMBEDDINGS)
# ==============================================================================
print("\n--- Initializing mBERT (Contextual Architecture) ---")
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
mbert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# ---------------------------------------------------------
# COMPUTE RESOURCE DETECTION & WORKLOAD SCALING
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cpu":
    print("CPU environment detected. Extreme downscaling to 112 sentences for a rapid functional test...")
    active_train_data = train_data.shuffle(seed=42).select(range(112))
    epochs = 1
else:
    print("GPU environment detected. Fast-tracking execution for live demonstration...")
    # Capped strictly at 7,500 sentences to guarantee sub-9-minute execution
    active_train_data = train_data.shuffle(seed=42).select(range(7500))
    epochs = 1

tokenized_train = active_train_data.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=epochs,
    per_device_train_batch_size=16,
    eval_strategy="no",
    logging_steps=50,
    save_strategy="no"
)

trainer = Trainer(
    model=mbert_model,
    args=training_args,
    train_dataset=tokenized_train,
)

print("Training mBERT sequence classifier...")
trainer.train()

print("\nEvaluating mBERT on Indonesian data (Zero-Shot Cross-Lingual Transfer)...")
predictions_output = trainer.predict(tokenized_test)
mbert_predictions = np.argmax(predictions_output.predictions, axis=1)

print("\n[mBERT Performance Metrics - Indonesian]")
print(classification_report(y_test_ft, mbert_predictions, zero_division=0))

# ==============================================================================
# 7. FUNCTIONAL INFERENCE PROTOTYPE (SINGLE-SAMPLE)
# ==============================================================================
print("\n" + "="*50)
print("FUNCTIONAL INFERENCE PROTOTYPE")
print("="*50)

def predict_emotion(text):
    print(f"\n[Input]: {text}")

    # ---------------------------------------------------------
    # 1. FastText Prediction
    # ---------------------------------------------------------
    # Reshape the vector because scikit-learn expects a 2D array for a single sample
    ft_vec = get_sentence_vector(text, ind_model).reshape(1, -1)
    ft_pred_id = fasttext_classifier.predict(ft_vec)[0]
    ft_emotion = ekman_id_to_string[ft_pred_id]

    # Get the maximum probability from the logistic regression
    ft_probs = fasttext_classifier.predict_proba(ft_vec)[0]
    ft_conf = np.max(ft_probs) * 100

    print(f"\n[FastText Baseline]")
    print(f"Prediction: {ft_emotion}")
    print(f"Confidence: {ft_conf:.2f}%")

    # ---------------------------------------------------------
    # 2. mBERT Prediction
    # ---------------------------------------------------------
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    mbert_model.to(device)

    with torch.no_grad():
        outputs = mbert_model(**inputs)

    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    confidence_score, predicted_class_id = torch.max(probabilities, dim=0)
    mbert_emotion = ekman_id_to_string[predicted_class_id.item()]

    print(f"\n[mBERT Contextual]")
    print(f"Prediction: {mbert_emotion}")
    print(f"Confidence: {confidence_score.item() * 100:.2f}%")

sample_ind_text = test_dataset['text'][0] if len(test_dataset) > 0 else "Saya sangat senang!"
predict_emotion(sample_ind_text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

...

Map:   0%|          | 0/796 [00:00<?, ? examples/s]

Training mBERT sequence classifier...


Step,Training Loss
50,1.307203
100,1.114895
150,1.079799
200,0.918298
250,0.991236
300,0.831874
350,0.846583
400,0.831466
450,0.749229



Evaluating mBERT on Indonesian data (Zero-Shot Cross-Lingual Transfer)...



[mBERT Performance Metrics - Indonesian]
              precision    recall  f1-score   support

           0       0.55      0.76      0.63       386
           1       0.36      0.33      0.34       178
           2       0.29      0.05      0.08        83
           3       0.00      0.00      0.00        44
           4       0.00      0.00      0.00        56
           5       0.29      0.49      0.36        49

    accuracy                           0.48       796
   macro avg       0.25      0.27      0.24       796
weighted avg       0.39      0.48      0.42       796


FUNCTIONAL INFERENCE PROTOTYPE

[Input]: berat ya kalau pacaran jarak jauh, gabisa ke kontrol takut selingkuh kalau hubungan jauh

[FastText Baseline]
Prediction: Joy
Confidence: 76.80%

[mBERT Contextual]
Prediction: Joy
Confidence: 63.53%
